In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1427_Najafgarh_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,178.88,279.76,3.10,14.60,17.78,5.09,2.76,0.51,13.21,...,NaN,12.40,75.66,0.28,150.52,0.0,0.0,16.03,995.51,NaN
1,2024-01-02,153.05,221.35,3.36,11.35,14.82,19.53,2.59,0.67,15.29,...,NaN,11.23,74.18,0.26,135.97,0.0,0.0,20.30,994.76,NaN
2,2024-01-03,173.24,274.54,3.80,13.13,16.91,38.16,2.97,0.51,12.24,...,NaN,10.77,81.64,0.31,245.92,0.0,0.0,16.36,994.59,NaN
3,2024-01-04,159.90,245.04,3.43,18.33,14.74,37.23,4.88,0.58,8.75,...,NaN,10.73,84.14,0.32,273.55,0.0,0.0,9.56,994.99,NaN
4,2024-01-05,111.13,163.12,2.41,20.13,12.67,33.42,7.82,0.49,32.40,...,NaN,12.86,81.75,0.34,207.50,0.0,0.0,10.76,995.03,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,246.77,317.50,38.95,62.24,64.77,71.25,13.38,2.27,20.38,...,NaN,16.92,82.20,0.63,171.20,0.0,0.0,4.33,996.53,NaN
362,2024-12-28,94.07,110.02,10.66,29.98,24.06,74.22,9.56,1.10,23.89,...,NaN,17.82,82.95,0.50,185.30,0.0,0.0,15.73,995.77,NaN
363,2024-12-29,69.95,84.68,8.16,30.71,22.97,41.53,10.26,1.09,45.37,...,NaN,16.33,81.72,0.64,254.67,0.0,0.0,18.88,997.45,NaN
364,2024-12-30,67.26,81.09,7.14,32.39,23.04,34.68,11.17,0.97,44.62,...,NaN,15.01,80.65,0.59,224.20,0.0,0.0,25.42,996.79,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 4
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (362, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         178.88        279.76        3.10        14.60   
1  2024-01-02         153.05        221.35        3.36        11.35   
2  2024-01-03         173.24        274.54        3.80        13.13   
3  2024-01-04         159.90        245.04        3.43        18.33   
4  2024-01-05         111.13        163.12        2.41        20.13   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      17.78         5.09       11.035        0.51          13.21   
1      14.82        19.53       11.035        0.67          15.29   
2      16.91        38.16       11.035        0.51          12.24   
3      14.74        37.23        4.880        0.58           8.75   
4      12.67        33.42        7.820        0.49          32.40   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             2.05             4.46    12.40   75.66      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.933100,1.470820,-0.844759,-0.317215,0.356389,-1.904199,0.039117,-1.215056,-1.331435,-0.002647,-0.175782,-1.752300,0.942203,-1.934259,-2.063347,0.0,0.0,-1.305246,1.300510
1,2024-01-02,1.450286,0.782383,-0.759318,-0.527215,0.032016,-0.903445,0.039117,-0.678408,-1.248907,-0.323994,-0.633021,-1.915368,0.836078,-2.079364,-2.526556,0.0,0.0,-1.203916,1.198392
2,2024-01-03,1.827677,1.409296,-0.614726,-0.412200,0.261050,0.387693,0.039117,-1.215056,-1.369922,0.110420,-0.257604,-1.979479,1.371004,-1.716603,0.973782,0.0,0.0,-1.297415,1.175246
3,2024-01-04,1.578326,1.061600,-0.736315,-0.076200,0.023249,0.323240,-2.735703,-0.980273,-1.508394,1.348202,0.069683,-1.985054,1.550268,-1.644050,1.853403,0.0,0.0,-1.458784,1.229708
4,2024-01-05,0.666719,0.096068,-1.071506,0.040108,-0.203593,0.059191,-1.410281,-1.282137,-0.570034,1.502925,0.445100,-1.688188,1.378891,-1.498946,-0.249347,0.0,0.0,-1.430307,1.235155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,2024-12-27,-0.224047,1.915635,-0.187521,2.761061,-0.334000,-0.031944,1.096299,0.126564,-1.046951,0.717409,2.996011,-1.122331,1.411159,0.605070,-1.404984,0.0,0.0,-1.582896,1.439390
358,2024-12-28,0.347834,-0.529784,1.639598,0.676569,1.044587,-0.031944,-0.625848,0.763834,-0.907685,-0.925032,-0.690778,-0.996895,1.464938,-0.338110,-0.956100,0.0,0.0,-1.312365,1.335911
359,2024-12-29,-0.103017,-0.828449,0.818052,0.723739,0.925138,0.621249,-0.310271,0.730294,-0.055424,-0.925032,-1.085447,-1.204562,1.376740,0.677622,1.252345,0.0,0.0,-1.237613,1.564655
360,2024-12-30,-0.153298,-0.870761,0.482861,0.832292,0.932809,0.146514,0.099978,0.327808,-0.085182,-0.913131,-1.107105,-1.388535,1.300015,0.314861,0.282310,0.0,0.0,-1.082414,1.474791


In [10]:
df.to_excel('najafgarh2024.xlsx', index=False)